In [4]:
import psycopg2
import pandas as pd

In [5]:
# environment variable can take two values: DEV (development), PRD (production)
env = 'DEV'

In [6]:
# for production we will use following database uri variable
DATABASE_URI = ''

In [8]:
try:
    if env == 'DEV':
        con = psycopg2.connect(host="localhost", user='postgres', database='value-investing-dev', port='5432', password='v,1846PSVv,1846PSV')
    elif env == 'PRD':
        con = psycopg2.connect(DATABASE_URI)

    #create cursor to execute sql statements
    cur = con.cursor()

    # read the tax rates from the damodaran file
    df = pd.read_csv('InterestCoverageRatios.csv')

    # first we will check if already entries are present in table; if yes we will delete all of them
    sql_select_query = """select * from public.dcf_interestcoverageratios"""

    # execute the sql query
    cur.execute(sql_select_query)

    ratios = cur.fetchall()

    if len(ratios) > 0:
        sql_delete_query = """delete from public.dcf_interestcoverageratios"""

        #delete all entries
        cur.execute(sql_delete_query)

        #commit deletion
        con.commit()
    

    # iterate through dataframe to create new entries or update existing ones
    for index, row in df.iterrows():
        #extract lower and upper bound
        lowerBound = float(row['>'])
        upperBound = float(row['≤ to'])
        rating = row['Rating is']

        #extract default spread of company; check if % sign is included
        default_spread = row["Spread is"]
        if '%' in default_spread:
            default_spread = float(default_spread.split('%')[0])/100
        else:
            default_spread = float(default_spread)/100

        sql_insert_query = """INSERT INTO public.dcf_interestcoverageratios("lowerBound", "upperBound", RATING, "companyDefaultSpread") VALUES(%s, %s, %s, %s)"""
        cur.execute(sql_insert_query, (lowerBound, upperBound, rating, default_spread))
        con.commit()
    
    cur.close()

except Exception as error:
    print('Could not connect to the database: ', error)
